# Stereo-seq (FF footpad tumors): melanoma signatures and Cxcl2 (Fig. 3C; Supplementary Figs. S7E, S8A, S10A)
AUCell scores of melanoma pseudospots in 50 µm layers from -150 µm (peritumor) to 700 µm (center).
Lines show the group mean of the per-sample layer means; shaded areas show mean ± SEM across samples (n = 3 per group).
- **Fig. 3C** YAP signature, EMT-like signature (Hallmark EMT), immune receptor activity (GO:MF)
- **Supplementary Fig. S7E** cell-matrix adhesion, chemokine signaling, IL-1β production, Tirosh EMT1
- **Supplementary Fig. S8A** Cxcl2 expression of melanoma bins per region
- **Supplementary Fig. S10A** Hallmark hypoxia

In [ ]:
import os
import re
import importlib.metadata
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
import anndata as ad
import decoupler as dc
from scipy import stats

from spatial_utils import *

%matplotlib inline
plt.rcParams['pdf.fonttype'] = 42
sc.settings.verbosity = 3
sc.set_figure_params(dpi = 100, dpi_save = 300, transparent=True, color_map = 'OrRd')
os.makedirs('./figures', exist_ok=True)

for package in ['stereopy', 'numpy', 'pandas', 'matplotlib', 'scanpy', 'anndata', 'squidpy', 'decoupler']:
    try:
        print(f"{package} version: {importlib.metadata.version(package)}")
    except importlib.metadata.PackageNotFoundError:
        print(f"{package} is not installed")

# Annotated bin30 objects: <GEO library name>_bin30_annotated.h5ad
SPATIAL_DIR = '../data/stereoseq_annotated'
sample_list = SAMPLE_LIST

In [ ]:
lognorm4 = load_scored(SPATIAL_DIR, sample_list)

## AUCell scores not stored in the h5ad files

In [ ]:
import json

# Signatures shown in Fig. 3C / Supplementary Fig. S7E that are not stored in the h5ad files.
# AUCell is run on the same normalised data as in the original analysis (min_n=1).
with open('../data/genesets_used.json', 'r') as f:
    genesets = json.load(f)
extra = {k: genesets[k] for k in ['GOMF_IMMUNE_RECEPTOR_ACTIVITY', 'WP_CHEMOKINE_SIGNALING_PATHWAY',
                                  'GOBP_INTERLEUKIN_1_BETA_PRODUCTION']}
tiroshi = pd.read_csv('../data/Tiroshi_EMT.csv')
extra['Tiroshi_EMT1'] = [x.lower().capitalize() for x in tiroshi['Tiroshi_EMT1'].dropna()]

for sample in sample_list:
    adata = lognorm4[sample]
    net = pd.DataFrame([(g, name) for name, genes in extra.items()
                        for g in set(x for x in genes if x in adata.var_names)], columns=['target', 'source'])
    dc.run_aucell(mat=adata, net=net, source='source', target='target', verbose=True, min_n=1)
    for name in extra:
        adata.obs[f'aucell_{name}'] = adata.obsm['aucell_estimate'][name].values

## Fig. 3C — YAP, EMT-like and immune receptor signatures

In [ ]:
# peritumor (-150-0 um), margin (0-150 um) and center (150-700 um) layers
layers = ['-150um~-100um', '-100um~-50um', '-50um~0um', '0um~50um', '50um~100um', '100um~150um', '150um~200um',
          '200um~250um', '250um~300um', '300um~350um', '350um~400um', '400um~450um', '450um~500um',
          '500um~550um', '550um~600um', '600um~650um', '650um~700um']

for term in ['aucell_YAP_3', 'aucell_HALLMARK_EPITHELIAL_MESENCHYMAL_TRANSITION', 'aucell_GOMF_IMMUNE_RECEPTOR_ACTIVITY']:
    exp_df = pd.DataFrame(index=sample_list, columns=layers)
    for sample in sample_list:
        adata = lognorm4[sample]
        for layer in layers:
            temp = adata[(adata.obs['tumor_layer'] == layer) & (adata.obs['cell_type'] == 'Melanoma')]
            exp_df.loc[sample, layer] = np.mean(temp.obs[term])
    plot_layer_profile(exp_df, term, figpath=f'./figures/Fig3C_{term}.pdf')

## Supplementary Fig. S7E — Cell-matrix adhesion, chemokine signaling, IL-1β production, Tirosh EMT1

In [ ]:
# peritumor (-150-0 um), margin (0-150 um) and center (150-700 um) layers
layers = ['-150um~-100um', '-100um~-50um', '-50um~0um', '0um~50um', '50um~100um', '100um~150um', '150um~200um',
          '200um~250um', '250um~300um', '300um~350um', '350um~400um', '400um~450um', '450um~500um',
          '500um~550um', '550um~600um', '600um~650um', '650um~700um']

for term in ['aucell_GOBP_REGULATION_OF_CELL_MATRIX_ADHESION', 'aucell_WP_CHEMOKINE_SIGNALING_PATHWAY',
             'aucell_GOBP_INTERLEUKIN_1_BETA_PRODUCTION', 'aucell_Tiroshi_EMT1']:
    exp_df = pd.DataFrame(index=sample_list, columns=layers)
    for sample in sample_list:
        adata = lognorm4[sample]
        for layer in layers:
            temp = adata[(adata.obs['tumor_layer'] == layer) & (adata.obs['cell_type'] == 'Melanoma')]
            exp_df.loc[sample, layer] = np.mean(temp.obs[term])
    plot_layer_profile(exp_df, term, figpath=f'./figures/FigS7E_{term}.pdf')

## Supplementary Fig. S10A — Hypoxia signature

In [ ]:
# Supplementary Fig. S10A: Hallmark hypoxia (AUCell with the default min_n, as in the original analysis)
hypoxia = pd.DataFrame({'target': genesets['HALLMARK_HYPOXIA'], 'source': 'HALLMARK_HYPOXIA'})
for sample in sample_list:
    adata = lognorm4[sample]
    net = hypoxia[hypoxia['target'].isin(adata.var_names)]
    dc.run_aucell(mat=adata, net=net, source='source', target='target', verbose=True)
    adata.obs['aucell_HALLMARK_HYPOXIA'] = adata.obsm['aucell_estimate']['HALLMARK_HYPOXIA'].values

In [ ]:
# peritumor (-150-0 um), margin (0-150 um) and center (150-700 um) layers
layers = ['-150um~-100um', '-100um~-50um', '-50um~0um', '0um~50um', '50um~100um', '100um~150um', '150um~200um',
          '200um~250um', '250um~300um', '300um~350um', '350um~400um', '400um~450um', '450um~500um',
          '500um~550um', '550um~600um', '600um~650um', '650um~700um']

for term in ['aucell_HALLMARK_HYPOXIA']:
    exp_df = pd.DataFrame(index=sample_list, columns=layers)
    for sample in sample_list:
        adata = lognorm4[sample]
        for layer in layers:
            temp = adata[(adata.obs['tumor_layer'] == layer) & (adata.obs['cell_type'] == 'Melanoma')]
            exp_df.loc[sample, layer] = np.mean(temp.obs[term])
    plot_layer_profile(exp_df, term, figpath=f'./figures/FigS10A_{term}.pdf')

## Supplementary Fig. S8A — Cxcl2 expression of melanoma bins

In [ ]:
# Mean log-normalised Cxcl2 of melanoma bins per 50 um layer, averaged per region (CXCL.ipynb); no statistics
box_df = region_mean_expression(lognorm4, 'Cxcl2', cell_type='Melanoma')
display(box_df.pivot(index='Sample', columns='Region', values='Value')[list(REGION_LAYERS)])
fig, ax = plot_region_box(box_df, 'Cxcl2')
plt.show()